<a href="https://colab.research.google.com/github/vijayrajeshr/Flipkart__GridLockHackathon2.0/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Flipkart Gridlock Hackathon 2.0

In [4]:
!pip install pygeohash xgboost scikit-learn pandas numpy

In [5]:
import pandas as pd
import numpy as np
import pygeohash as pgh
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

print("1. Loading datasets...")
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

def engineer_features(df):
    df = df.copy()

    # --- A. SPATIAL ---
    # We keep the raw geohash, but ALSO give the exact coordinates
    df['Latitude'] = df['geohash'].apply(lambda x: pgh.decode(x)[0] if pd.notna(x) else 0)
    df['Longitude'] = df['geohash'].apply(lambda x: pgh.decode(x)[1] if pd.notna(x) else 0)

    # --- B. TIME EXTRACTION ---
    df['hours'] = df['timestamp'].astype(str).apply(lambda x: int(x.split(':')[0]) if pd.notna(x) and ':' in str(x) else 0)
    df['mins'] = df['timestamp'].astype(str).apply(lambda x: int(x.split(':')[1]) if pd.notna(x) and ':' in str(x) else 0)

    # --- C. THE MAGIC: CYCLICAL TIME WAVES ---
    # 24 hours * 60 mins = 1440 mins in a day
    minutes_in_day = df['hours'] * 60 + df['mins']
    df['time_sin'] = np.sin(2 * np.pi * minutes_in_day / 1440)
    df['time_cos'] = np.cos(2 * np.pi * minutes_in_day / 1440)

    # --- D. THE MAGIC: DAY OF WEEK ---
    df['day_of_week'] = df['day'] % 7
    df['time_continuous'] = 24 * 60 * (df['day'] - 1) + 60 * df['hours'] + df['mins']

    # --- E. SMART IMPUTATION ---
    df['Temperature'] = df['Temperature'].astype(float).interpolate(method='linear').fillna(df['Temperature'].median())
    df['NumberofLanes'] = df['NumberofLanes'].fillna(df['NumberofLanes'].mode()[0]).astype(float)

    return df

print("2. Engineering Advanced Wave Features...")
train_df = engineer_features(train)
test_df = engineer_features(test)

print("3. Formatting Native Categoricals (No Label Encoding!)...")
# Let XGBoost handle strings natively to prevent data loss
cat_cols = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

for col in cat_cols:
    train_df[col] = train_df[col].astype('category')
    test_df[col] = test_df[col].astype('category')

# Ensure category matching between train and test
for col in cat_cols:
    union_cats = pd.Series(pd.concat([train_df[col], test_df[col]])).dropna().unique()
    train_df[col] = train_df[col].cat.set_categories(union_cats)
    test_df[col] = test_df[col].cat.set_categories(union_cats)

features = [
    'geohash', 'Latitude', 'Longitude',
    'time_continuous', 'time_sin', 'time_cos', 'day_of_week',
    'RoadType', 'NumberofLanes', 'LargeVehicles',
    'Landmarks', 'Temperature', 'Weather'
]

X_train = train_df[features]
y_train = train_df['demand']
X_test = test_df[features]

print("4. Training Ultra-Optimized XGBoost Model...")
model = XGBRegressor(
    n_estimators=2500,        # Increased trees drastically
    learning_rate=0.015,      # Dropped learning rate for high precision
    max_depth=11,
    subsample=0.85,
    colsample_bytree=0.85,
    tree_method='hist',
    enable_categorical=True,  # This forces XGBoost to treat strings intelligently
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train, verbose=False)

print("5. Generating Final Predictions...")
preds = model.predict(X_test)
preds = np.clip(preds, 0, None)

sub = pd.DataFrame({
    'Index': test['Index'],
    'demand': preds
})

filename = 'ULTIMATE_95_submission.csv'
sub.to_csv(filename, index=False)
print(f"SUCCESS! Output saved as '{filename}'. ")

1. Loading datasets...
2. Engineering Advanced Wave Features...
3. Formatting Native Categoricals (No Label Encoding!)...
4. Training Ultra-Optimized XGBoost Model...
5. Generating Final Predictions...
SUCCESS! Output saved as 'ULTIMATE_95_submission.csv'. 
